In [25]:
import requests
import pandas as pd
import trueskill
from datetime import datetime
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import json
import sqlite3
from xgboost import XGBClassifier

def get_log_data(log_id):
    url = f"https://logs.tf/api/v1/log/{log_id}"
    response = requests.get(url)
    if response.status_code == 429:
        print("Rate limited. Sleeping...")
        time.sleep(2)
        response = requests.get(url)
    elif response.status_code != 200:
        raise Exception("Failed to fetch data")
    return response.json()

def parse_players(log_json):
    return log_json["players"].items()
    



In [228]:
log = get_log_data(4040975)
player_info = parse_players(log)
for user, p in player_info :
    print(p['class_stats'][0]['type'])

print(log)

{'version': 3, 'teams': {'Red': {'score': 3, 'kills': 0, 'deaths': 0, 'dmg': 0, 'charges': 0, 'drops': 0, 'firstcaps': 3, 'caps': 3}, 'Blue': {'score': 0, 'kills': 0, 'deaths': 0, 'dmg': 0, 'charges': 0, 'drops': 0, 'firstcaps': 0, 'caps': 0}}, 'length': 673, 'players': {}, 'names': {}, 'rounds': [{'start_time': 1776000724, 'winner': 'Red', 'team': {'Blue': {'score': 0, 'kills': 0, 'dmg': 0, 'ubers': 0}, 'Red': {'score': 1, 'kills': 0, 'dmg': 0, 'ubers': 0}}, 'events': [{'type': 'pointcap', 'time': 48, 'team': 'Red', 'point': 1}, {'type': 'round_win', 'time': 228, 'team': 'Red'}], 'players': {'[U:1:269483640]': {'team': 'Red', 'kills': 0, 'dmg': 0}, 'BOT': {'team': 'Blue', 'kills': 0, 'dmg': 0}}, 'firstcap': 'Red', 'length': 228}, {'start_time': 1776000962, 'winner': 'Red', 'team': {'Blue': {'score': 0, 'kills': 0, 'dmg': 0, 'ubers': 0}, 'Red': {'score': 2, 'kills': 0, 'dmg': 0, 'ubers': 0}}, 'events': [{'type': 'pointcap', 'time': 280, 'team': 'Red', 'point': 1}, {'type': 'round_win',

In [229]:
def get_match_outcome(log_json):
    teams = log_json["teams"]
    if teams['Red']['score'] > teams['Blue']['score']:
        return [str(teams['Red']['score']), str(teams['Blue']['score']), [1, 0]]
    if teams['Red']['score'] < teams['Blue']['score']:
        return [str(teams['Red']['score']), str(teams['Blue']['score']), [0, 1]]
    return [str(teams['Red']['score']), str(teams['Blue']['score']), [0, 0]]
    

In [230]:
def get_last_n_logs(n=100, max_retries=3):
    """
    Retrieve the last n logs from logs.tf (excluding passtime matches)
    
    Args:
        n: Number of logs to retrieve (can be > 1000)
        max_retries: Number of retries for failed requests
    
    Returns:
        List of log IDs
    """
    base_url = "https://logs.tf/api/v1/log"
    all_logs = []
    offset = 0
    limit = 1000  # Max per request
    
    # Keep fetching until we have enough logs
    while len(all_logs) < n:
        params = {
            "limit": limit,
            "offset": offset
        }
        
        # Make request with retries
        for attempt in range(max_retries):
            try:
                response = requests.get(base_url, params=params, timeout=10)
                response.raise_for_status()
                break  # Success, exit retry loop
            except requests.exceptions.RequestException as e:
                if attempt == max_retries - 1:
                    print(f"Error fetching logs at offset {offset}: {e}")
                    return []  # Return what we have
                time.sleep(1)  # Wait before retry
        
        data = response.json()
        logs_batch = data.get("logs", [])
        
        # If no logs returned, we've reached the end
        if not logs_batch:
            break
        
        all_logs.extend(logs_batch)
        offset += limit
        
        # Be respectful of API rate limits
        time.sleep(0.5)
        
        # Optional: Print progress
        print(f"Fetched {len(all_logs)} logs so far...")
    
    # Sort by date descending (newest first)
    all_logs_sorted = sorted(all_logs, key=lambda x: x["date"], reverse=True)
    
    # Filter out passtime logs and take first n
    last_n_log_ids = []
    for log in all_logs_sorted:
        if len(last_n_log_ids) >= n:
            break
            
        if re.search('passtime', log["title"], re.IGNORECASE):
            continue
            
        last_n_log_ids.append(log["id"])
    
    print(f"Retrieved {len(last_n_log_ids)} logs (filtered out passtime matches)")
    return last_n_log_ids

def train_test_split(last_n_logs, p):
    n = len(last_n_logs)
    m = int(n*p)
    test, train = last_n_logs[:m], last_n_logs[m:]
    print([len(train), len(test)])

    return train, test

In [231]:
conn = sqlite3.connect("tf2.db")  # creates file
cursor = conn.cursor()
cursor.execute("DROP TABLE IF EXISTS player_skills")
cursor.execute("DROP TABLE IF EXISTS match_outcomes")
cursor.execute('''CREATE TABLE IF NOT EXISTS match_outcomes (
                    log_id integer PRIMARY KEY,
                    red_score integer,
                    blue_score integer,
                    red_team_ids text,
                    blue_team_ids text,
                    diff_ts_score real,
                    diff_avg_kills real, 
                    diff_avg_assists real, 
                    diff_avg_deaths real,
                    diff_avg_healing real,
                    outcome integer
               )
               ''')
cursor.execute('''CREATE TABLE IF NOT EXISTS PLAYER_SKILLS (
                    player_id text primary key, 
                    mu REAL, 
                    sigma REAL, 
                    total_assists real,
                    total_healing real,
                    total_kills integer,
                    total_deaths integer,
                    matches_played integer
               ) 
               ''')
conn.commit()

In [232]:
def get_player_skills(cursor, player_id):
    cursor.execute("""
    select
        mu,
        sigma, 
        total_assists, 
        total_healing, 
        total_kills,
        total_deaths,          
        matches_played
    from player_skills
    where player_id = ?
    """, (player_id,)
    ) 
    return cursor.fetchone()

def update_player_skills(cursor, player_id, mu, sigma, deaths, kills, assists, matches_played):
    cursor.execute("""
    update player_skills  
    set 
        mu = ?, 
        sigma = ?,
        matches_played = ?, 
        total_deaths = ?, 
        total_kills = ?, 
        total_assists = ?,           
        total_healing = ?
                   
    where player_id = ?""", (mu, sigma, matches_played, deaths, kills, assists, player_id)
    )

def add_match_outcome(cursor, outcome, log_id, red_ids, blue_ids):
    # outcome format from get_match_outcome(): [red_score, blue_score, [rank_red, rank_blue]]
    # where [1, 0] means Red won, [0, 1] means Blue won
    if outcome[2] == [1, 0]:
        cursor.execute("""INSERT INTO match_outcomes
                            (log_id, red_score, blue_score, red_team_ids, blue_team_ids, outcome)
                        values (?, ?, ?, ?, ?, 1)""",
                       (log_id, outcome[0], outcome[1], ','.join(red_ids), ','.join(blue_ids) )
        )
    elif outcome[2] == [0, 1]:
        cursor.execute("""INSERT INTO match_outcomes
                            (log_id, red_score, blue_score, red_team_ids, blue_team_ids, outcome)
                        values (?, ?, ?, ?, ?, 0)""",
                       (log_id, outcome[0], outcome[1], ','.join(red_ids), ','.join(blue_ids) )
        )
    # Draws are already filtered out earlier, but just in case
    else:
        # outcome[2] == [0, 0] - draw, skip insertion
        pass
    #insert blue team and red team player_ids, 

def insert_default_player_stats(cursor, player_id):
    cursor.execute("""INSERT INTO player_skills 
                               (player_id, mu, sigma, total_assists, total_healing, total_kills, total_deaths, matches_played)
                               VALUES
                                (?, 25.0, 25.0/3, 0, 0, 0, 0, 0)
                    """, (player_id,)
    )

In [233]:
class StreamingPlayerSkillProcessor:
    def __init__(self, cursor, connection, env):
        self.cursor = cursor
        self.connection = connection
        self.env = env
        self.batch_update_buffer = []  # Buffer for batch updates
        self.batch_size = 100  # Update every 100 matches
        self.player_cache = {}  # {player_id: (mu, sigma, matches_played)}
        self.X_test = pd.DataFrame(columns = ['diff_ts_score', 'diff_avg_kills', 'diff_avg_assists', 'diff_avg_deaths', 'diff_avg_healing'])
        self.y_test = pd.Series(name = 'outcome')

        
    def get_player_skills_streaming(self, train):
        """Process logs sequentially with batching and caching"""
        
    
        for log_id in train : 
            try:
                # Fetch and parse log
                log = get_log_data(log_id)
                
                players = parse_players(log)
                outcome = get_match_outcome(log)

                # Skip draws
                if outcome[2] == [0, 0]:
                    print(f"Log {log_id} was a draw, skipping")
                    continue
                
                # Process the match
                red_ids, blue_ids = self._process_match_streaming(players, outcome)

                # Insert match outcome
                add_match_outcome(self.cursor, outcome, log_id, red_ids, blue_ids)
                
                if len(self.batch_update_buffer) > self.batch_size :
                    self._flush_updates()


                    
            except Exception as e:
                print(f"Error processing log {log_id}: {e}")
                import traceback
                traceback.print_exc()
                self.connection.rollback()
                continue
    
            
        # Final flush for remaining updates
        if self.batch_update_buffer:
            self._flush_updates()
        
        self.connection.commit()
        print("All logs processed successfully")
    
    def _process_match_streaming(self, players, outcome):
        """Process a single match with caching"""
        red_team_scores = []
        blue_team_scores = []
        red_team_stats = []
        blue_team_stats = []
        red_team_ids = []
        blue_team_ids = []

        
        # Parse players - players is a dict_items of (player_id, player_data)
        for player_id, player_data in players:
            # Check cache first
            assists, healing, kills, deaths = player_data['assists'], player_data['heal'], player_data['kills'], player_data['deaths'], 
            if player_id in self.player_cache:
                curr_mu, curr_sigma, curr_assists, curr_healing, curr_kills, curr_deaths, curr_matches = self.player_cache[player_id]
            else:
                # Try to get from database
                skills = get_player_skills(self.cursor, player_id)
                if skills:
                    curr_mu, curr_sigma, curr_assists, curr_healing, curr_kills, curr_deaths, curr_matches = skills
                else:
                    # New player - insert with defaults
                    insert_default_player_stats(self.cursor, player_id)
                    curr_mu, curr_sigma, curr_assists, curr_healing, curr_kills, curr_deaths, curr_matches = 25.0, 25.0/3, 0, 0, 0, 0, 0
                    self.connection.commit()
                
                # Cache the result
                self.player_cache[player_id] = (curr_mu, curr_sigma, curr_assists, curr_healing, curr_kills, curr_deaths, curr_matches)
            
            # Get team from player data
            team = player_data.get('team', 'Unknown')
            new_assists = curr_assists + assists
            new_healing = curr_healing + healing
            new_kills = curr_kills + kills
            new_deaths = curr_deaths + deaths
            new_matches = curr_matches + 1
            # Assign to teams
            if team == 'Red':
                red_team_scores.append(self.env.Rating(mu=curr_mu, sigma=curr_sigma))
                red_team_stats.append((player_id, new_assists, new_healing, new_kills, new_deaths, new_matches))
                red_team_ids.append(player_id)
            elif team == 'Blue':
                blue_team_scores.append(self.env.Rating(mu=curr_mu, sigma=curr_sigma))
                blue_team_stats.append((player_id, new_assists, new_healing, new_kills, new_deaths, new_matches))
                blue_team_ids.append(player_id)
        # Check if both teams have players
        if len(red_team_ids) == 0 or len(blue_team_ids) == 0:
            print(f"Warning: Missing team players - Red: {len(red_team_ids)}, Blue: {len(blue_team_ids)}")
            return
        
        # Rate the match using TrueSkill
        # outcome[2] contains [1, 0] for Red win, [0, 1] for Blue win
        try:
            new_red_ratings, new_blue_ratings = self.env.rate(
                [red_team_scores, blue_team_scores],
                ranks=outcome[2]
            )
        except Exception as e:
            print(f"Error rating match: {e}")
            return
        
        
        # Prepare updates for Red team using update_player_skills format
        for i, (player_id, new_assists, new_healing, new_kills, new_deaths, new_matches) in enumerate(red_team_stats):
            new_rating = new_red_ratings[i]
            new_mu, new_sigma = new_rating.mu, new_rating.sigma
            
            # Buffer for database update
            self.batch_update_buffer.append((new_mu, new_sigma, new_assists, new_healing, new_kills, new_deaths, new_matches, player_id))
            
            # Update cache immediately for subsequent matches
            self.player_cache[player_id] = (new_mu, new_sigma, new_assists, new_healing, new_kills, new_deaths, new_matches)
        
        # Prepare updates for Blue team
        for i, (player_id, new_assists, new_healing, new_kills, new_deaths, new_matches) in enumerate(blue_team_stats):
            new_rating = new_blue_ratings[i]
            new_mu, new_sigma = new_rating.mu, new_rating.sigma
            
            self.batch_update_buffer.append((new_mu, new_sigma, new_assists, new_healing, new_kills, new_deaths, new_matches, player_id))
            self.player_cache[player_id] = (new_mu, new_sigma, new_assists, new_healing, new_kills, new_deaths, new_matches)
    
        return red_team_ids, blue_team_ids
    
    def _flush_updates(self):
        """Execute batched updates"""
        if not self.batch_update_buffer:
            return
        
        try:
            # Use executemany for batch updates
            # SQLite uses ? for placeholders
            self.cursor.executemany("""
                UPDATE player_skills 
                SET mu = ?, sigma = ?, total_assists = ?, total_healing = ?, total_kills = ?, total_deaths = ?,  matches_played = ?
                WHERE player_id = ?
            """, self.batch_update_buffer)
            
            # Commit transaction
            self.connection.commit()
            
            print(f"Flushed {len(self.batch_update_buffer)} updates to database")
            self.batch_update_buffer = []
            
        except Exception as e:
            print(f"Error flushing updates: {e}")
            self.connection.rollback()
            raise

    def get_test_data(self, test):
        ds = {
            'mu': 25.0, 
            'sigma': 25.0/3,
            'kills_per_match': 12.0,
            'deaths_per_match': 12.0,  
            'assists_per_match': 8.0,
            'healing_per_match': 3000.0
        }   
        
        for log_id in test:
            try:
                log = get_log_data(log_id)
                outcome = get_match_outcome(log)

                # Skip draws
                if outcome[2] == [0, 0]:
                    print(f"Log {log_id} was a draw, skipping")
                    continue
                    
                players = parse_players(log)
                
                # Initialize accumulators
                blue_ts_score = 0
                red_ts_score = 0
                avg_blue_kills = 0
                avg_red_kills = 0
                avg_blue_deaths = 0
                avg_red_deaths = 0
                avg_blue_healing = 0
                avg_red_healing = 0
                avg_blue_assists = 0
                avg_red_assists = 0
                red_player_count = 0
                blue_player_count = 0
                
                # Process all players first
                for player_id, p in players:
                    # Try cache first, then database
                    if player_id in self.player_cache:
                        mu, sigma, assists, kills, deaths, healing, matches = self.player_cache[player_id]
                    else:
                        skills = get_player_skills(self.cursor, player_id)
                        if skills:
                            mu, sigma, assists, kills, deaths, healing, matches = skills
                        else:
                            # Cold start defaults
                            mu, sigma = ds['mu'], ds['sigma']
                            assists = ds['assists_per_match']
                            kills = ds['kills_per_match']
                            deaths = ds['deaths_per_match']
                            healing = ds['healing_per_match']
                            matches = 1
                    
                    ts_score = mu - 3 * sigma
                    avg_kills = kills / matches if matches > 0 else 0
                    avg_deaths = deaths / matches if matches > 0 else 0
                    avg_assists = assists / matches if matches > 0 else 0
                    avg_healing = healing / matches if matches > 0 else 0
                    
                    team = p.get('team')
                    if team == 'Red':
                        red_ts_score += ts_score
                        avg_red_kills += avg_kills
                        avg_red_deaths += avg_deaths
                        avg_red_assists += avg_assists
                        avg_red_healing += avg_healing  # Add for ALL players, not just medics
                        red_player_count += 1
                    elif team == 'Blue':
                        blue_ts_score += ts_score
                        avg_blue_kills += avg_kills
                        avg_blue_deaths += avg_deaths
                        avg_blue_assists += avg_assists
                        avg_blue_healing += avg_healing
                        blue_player_count += 1
                
                # Check for empty teams
                if red_player_count == 0 or blue_player_count == 0:
                    print(f"Log {log_id}: Missing team - Red: {red_player_count}, Blue: {blue_player_count}")
                    continue
                
                # Calculate differences (AFTER the player loop)
                diff_ts_score = (red_ts_score / red_player_count) - (blue_ts_score / blue_player_count)
                diff_avg_kills = (avg_red_kills / red_player_count) - (avg_blue_kills / blue_player_count)
                diff_avg_deaths = (avg_red_deaths / red_player_count) - (avg_blue_deaths / blue_player_count)
                diff_avg_healing = (avg_red_healing / red_player_count) - (avg_blue_healing / blue_player_count)
                diff_avg_assists = (avg_red_assists / red_player_count) - (avg_blue_assists / blue_player_count)
                
                # Target: 1 if Red won, 0 if Blue won
                target = 1 if outcome[2] == [1, 0] else 0
                
                # Create row (all variables now defined)
                row = {
                    'diff_ts_score': diff_ts_score,
                    'diff_avg_assists': diff_avg_assists,
                    'diff_avg_kills': diff_avg_kills,
                    'diff_avg_healing': diff_avg_healing,
                    'diff_avg_deaths': diff_avg_deaths,
                }
                
                self.X_test.loc[log_id] = row
                self.y_test[log_id] = target
                
            except Exception as e:
                print(f"Error processing test log {log_id}: {e}")
                import traceback
                traceback.print_exc()
                continue
        
        return

        



# Example usage
def process_matches(n):
    import trueskill
    
    # Setup your database connection here
    # conn = your_connection
    # cursor = conn.cursor()
    
    env = trueskill.TrueSkill()
    processor = StreamingPlayerSkillProcessor(cursor, conn, env)
    
    # Get last 10 logs to test
    train, test = train_test_split(get_last_n_logs(n), 0.2)
    processor.get_player_skills_streaming(train)
    processor.get_test_data(test)
    return processor.X_test, processor.y_test

In [ ]:
X_test, y_test = process_matches(500000)

In [235]:
# conn = sqlite3.connect("tf2.db")
# df = pd.read_sql_query("""select * from player_skills""", conn)
# df.head()

In [236]:
class complete_outcomes_table:
    def __init__(self, cursor, connection):
        self.cursor = cursor
        self.connection = connection
        self.cursor = cursor
        self.batch_update_buffer = []
        self.batch_size = 100

    def complete(self, outcomes):

        for row in outcomes :
            log_id, red_ids, blue_ids = row
            red_players = red_ids.split(',')
            blue_players = blue_ids.split(',')
            blue_ts_score = 0
            red_ts_score = 0
            avg_blue_kills = 0
            avg_red_kills = 0
            avg_blue_deaths = 0
            avg_red_deaths = 0
            avg_blue_healing = 0
            avg_red_healing = 0
            avg_blue_assists = 0
            avg_red_assists = 0
            red_player_count = 0
            blue_player_count = 0
            for player_id in red_players : 
                (mu, sigma, assists, healing, kills, deaths, matches) = get_player_skills(self.cursor, player_id)
                ts_score = mu-3*sigma
                avg_healing = healing/matches
                avg_kills = kills/matches
                avg_deaths = deaths/matches
                avg_assists = assists/matches
                red_ts_score += ts_score
                avg_red_kills += avg_kills
                avg_red_deaths += avg_deaths
                avg_red_assists += avg_assists
                avg_red_healing+=avg_healing
                red_player_count += 1

            for player_id in blue_players :
                (mu, sigma, assists, healing, kills, deaths, matches) = get_player_skills(self.cursor, player_id)
                ts_score = mu-3*sigma
                avg_healing = healing/matches
                avg_kills = kills/matches
                avg_deaths = deaths/matches
                avg_assists = assists/matches
                blue_ts_score += ts_score
                avg_blue_kills += avg_kills
                avg_blue_deaths += avg_deaths
                avg_blue_assists += avg_assists
                avg_blue_healing+=avg_healing
                blue_player_count += 1
                
            if red_player_count == 0 or blue_player_count == 0:
                continue
            diff_ts_score = red_ts_score/red_player_count-blue_ts_score/blue_player_count
            diff_avg_kills = avg_red_kills/red_player_count - avg_blue_kills/blue_player_count
            diff_avg_deaths = avg_red_deaths/red_player_count - avg_blue_deaths/blue_player_count
            diff_avg_healing = avg_red_healing/red_player_count - avg_blue_healing/blue_player_count
            diff_avg_assists = avg_red_assists/red_player_count-avg_blue_assists/blue_player_count
            self.batch_update_buffer.append((diff_ts_score, diff_avg_kills, diff_avg_deaths, diff_avg_healing, diff_avg_assists, log_id))
            if len(self.batch_update_buffer) >= self.batch_size:
                self._flush_updates()
        # Final flush for remaining updates
        if self.batch_update_buffer:
            self._flush_updates()
        
        self.connection.commit()
        print("All logs processed successfully")

    def _flush_updates(self):
        """Execute batched updates"""
        if not self.batch_update_buffer:
            return
        
        try:
            # Use executemany for batch updates
            # SQLite uses ? for placeholders
            self.cursor.executemany("""
                update match_outcomes
                set 
                    diff_ts_score = ?, 
                    diff_avg_kills = ?, 
                    diff_avg_deaths = ?, 
                    diff_avg_healing = ?, 
                    diff_avg_assists = ?
                where log_id = ?""", self.batch_update_buffer
                    
            )
            
            # Commit transaction
            self.connection.commit()
            
            print(f"Flushed {len(self.batch_update_buffer)} updates to database")
            self.batch_update_buffer = []
            
        except Exception as e:
            print(f"Error flushing updates: {e}")
            self.connection.rollback()
            raise
            
       
                    


In [237]:
conn = sqlite3.connect("tf2.db")
cursor = conn.cursor()
log_ids_df = pd.read_sql_query("""select log_id, red_team_ids, blue_team_ids from match_outcomes""", conn)
log_ids = list(log_ids_df.itertuples(index = False))
Completion = complete_outcomes_table(cursor, conn)
Completion.complete(log_ids)

Flushed 100 updates to database
Flushed 100 updates to database
Flushed 100 updates to database
Flushed 100 updates to database
Flushed 100 updates to database
Flushed 100 updates to database
Flushed 100 updates to database
Flushed 100 updates to database
Flushed 100 updates to database
Flushed 100 updates to database
Flushed 100 updates to database
Flushed 100 updates to database
Flushed 100 updates to database
Flushed 100 updates to database
Flushed 100 updates to database
Flushed 100 updates to database
Flushed 100 updates to database
Flushed 100 updates to database
Flushed 100 updates to database
Flushed 100 updates to database
Flushed 100 updates to database
Flushed 100 updates to database
Flushed 100 updates to database
Flushed 100 updates to database
Flushed 100 updates to database
Flushed 100 updates to database
Flushed 100 updates to database
Flushed 100 updates to database
Flushed 100 updates to database
Flushed 100 updates to database
Flushed 100 updates to database
Flushed 

In [28]:
conn = sqlite3.connect("tf2.db")
cursor = conn.cursor()
df = pd.read_sql_query("""select * from match_outcomes""", conn)
df.index = df['log_id']
df.drop(['log_id'], axis = 1, inplace = True)



In [29]:
conn = sqlite3.connect("tf2.db")
cursor = conn.cursor()
df2 = pd.read_sql_query("""select * from player_skills limit 100""", conn)


conn.close()

In [30]:
df2.head(30)

,player_id,mu,sigma,total_assists,total_healing,total_kills,total_deaths,matches_played
0,[U:1:10403381],5.166558,1.403124,19153.0,801529.0,36569,26754,1932
1,[U:1:185989865],6.923912,1.304681,15686.0,106715195.0,28287,20017,1519
2,[U:1:59056350],10.109869,1.348018,7276.0,178920.0,14679,11681,722
3,[U:1:75438474],11.512906,1.404572,8750.0,365574.0,19037,18981,1032
4,[U:1:93355936],5.675409,1.435271,4939.0,372.0,11805,9226,618
5,[U:1:16284945],12.303614,1.365784,10438.0,2569973.0,21022,20516,1454
6,[U:1:197717452],13.078169,1.316092,20755.0,34741582.0,16510,22490,1942
7,[U:1:89112868],11.375466,1.308469,10995.0,500726.0,30112,30949,1536
8,[U:1:159413592],12.408049,1.332336,13237.0,3840602.0,30007,28320,1733
9,[U:1:14228231],13.354952,1.510137,3290.0,3644849.0,4581,4345,340


In [11]:
print(len(df))
print(len(X_test))

363026


NameError: name 'X_test' is not defined

In [241]:
X_test = X_test[["diff_ts_score", "diff_avg_kills", "diff_avg_assists", "diff_avg_deaths"]]
X_test.head()

,diff_ts_score,diff_avg_kills,diff_avg_assists,diff_avg_deaths
4041676,-8.784297,-4237.770664,1.218991,16.707211
4041675,-2.081006,-1067.476123,-0.832955,0.649811
4041674,-2.081006,-1067.476123,-0.832955,0.649811
4041673,-1.272573,2254.056652,0.883467,-3.603743
4041671,-12.769652,393.917996,-0.067491,0.541310


In [12]:
X_train, y_train = df[["diff_ts_score", "diff_avg_kills", "diff_avg_assists", "diff_avg_deaths"]][int(0.8*len(df)):], df['outcome'][int(0.8*len(df)):]
X_test, y_test = df[["diff_ts_score", "diff_avg_kills", "diff_avg_assists", "diff_avg_deaths"]][:int(0.8*len(df))], df['outcome'][:int(0.8*len(df))]

In [13]:
y_train.head()

log_id
3858750    1
3858751    1
3858752    0
3858753    1
3858756    0
Name: outcome, dtype: int64

In [23]:
print(len(df[(df['diff_ts_score'] < 0 ) & (df['diff_avg_deaths'] > 0 ) & (df['outcome'] == 1)])/len(df[df['outcome'] == 1]))

0.31021908240310475


In [16]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
# Create pipeline with scaling and logistic regression
pipeline = Pipeline([
    ('scaler', StandardScaler()),  # Standardize features
    ('classifier', XGBClassifier(
        n_estimators=5,
        max_depth=3,
        learning_rate=0.1,
        objective='binary:logistic')
        
    )
])
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")
conn.close()


Accuracy: 0.7144


In [26]:
import joblib
model = joblib.dump(pipeline, "TF2_prediction.pkl")

In [52]:
import typing_extensions

In [27]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import List
import joblib
import sqlite3

app = FastAPI()

pipeline = joblib.load("TF2_prediction.pkl")

# 1. Input validation ONLY
class MatchInput(BaseModel):
    blue_team: List[int]
    red_team: List[int]


# 2. Feature engineering (separate function)
def get_features(blue_players, red_players, cursor):
    blue_ts_score = 0
    red_ts_score = 0
    avg_blue_kills = 0
    avg_red_kills = 0
    avg_blue_deaths = 0
    avg_red_deaths = 0
    avg_blue_assists = 0
    avg_red_assists = 0

    if not blue_players:
        raise HTTPException(status_code=400, detail="Missing blue team")
    if not red_players:
        raise HTTPException(status_code=400, detail="Missing red team")

    for player_id in red_players:
        mu, sigma, assists, _, kills, deaths, matches = get_player_skills(cursor, player_id)

        if matches == 0:
            continue  # or handle differently

        red_ts_score += mu - 3 * sigma
        avg_red_kills += kills / matches
        avg_red_deaths += deaths / matches
        avg_red_assists += assists / matches

    for player_id in blue_players:
        mu, sigma, assists, _, kills, deaths, matches = get_player_skills(cursor, player_id)

        if matches == 0:
            continue

        blue_ts_score += mu - 3 * sigma
        avg_blue_kills += kills / matches
        avg_blue_deaths += deaths / matches
        avg_blue_assists += assists / matches

    n_red = len(red_players)
    n_blue = len(blue_players)

    return (
        red_ts_score/n_red - blue_ts_score/n_blue,
        avg_red_kills/n_red - avg_blue_kills/n_blue,
        avg_red_assists/n_red - avg_blue_assists/n_blue,
        avg_red_deaths/n_red - avg_blue_deaths/n_blue
    )


# 3. API route
@app.post("/predict")
def predict(data: MatchInput):
    conn = sqlite3.connect("tf2.db")
    cursor = conn.cursor()

    features = get_features(data.blue_team, data.red_team, cursor)

    conn.close()

    prediction = pipeline.predict([features])[0]

    return {"prediction": int(prediction)}